# 01 — Phase 1 LoRA retrain (Drive-checkpointed)
Clones pes-mtech-project, trains the paper-closer LoRA config (r=16, alpha=128, 5 epochs, 4000 samples), and streams every checkpoint + the best adapter to Google Drive so a disconnect never loses the best params.

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
!rm -rf /content/repo
!git clone https://github.com/kvarun314/pes-mtech-project.git /content/repo
%cd /content/repo
!pip install -q -e ".[colab]"
!pip install -q kaggle

In [ ]:
import os
from huggingface_hub import login

# Colab Secrets panel (recommended): add a secret named HF_TOKEN, then
# grant this notebook access to it (toggle in the Secrets panel) — it
# reaches this cell via google.colab.userdata, NOT via os.environ.
# Accept the Llama-3 license at https://huggingface.co/meta-llama/Meta-Llama-3-8B
# before this token will work.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")  # raises if access wasn't granted — fix that, don't swallow it
except ImportError:
    hf_token = os.environ.get("HF_TOKEN", "")

if not hf_token:
    raise RuntimeError(
        "HF_TOKEN is empty. Set it in Colab Secrets (left sidebar) and grant this "
        "notebook access, or set os.environ['HF_TOKEN'] directly."
    )
login(hf_token)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_RUNS_DIR = "/content/drive/MyDrive/pes-mtech-project/runs"

## Download the Kaggle dataset
Get `kaggle.json` from https://www.kaggle.com/settings -> Create New Token,
then upload it with the file-upload button in the Colab sidebar (it lands in `/content/`).
The next cell moves it into place and downloads the dataset.

In [ ]:
import os, shutil

os.makedirs("/root/.kaggle", exist_ok=True)
if os.path.exists("/content/kaggle.json"):
    shutil.move("/content/kaggle.json", "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets download -d arhamrumi/amazon-product-reviews -p /content/data --unzip
DATA_PATH = "/content/data/Reviews.csv"

In [ ]:
from agentic_sentiment.train.run_dir import RunDir
from agentic_sentiment.train.phase1_train import run_training

run_dir = RunDir(base_dir=DRIVE_RUNS_DIR)
print("Run directory:", run_dir.path)
metrics = run_training(DATA_PATH, run_dir)
print(metrics)

In [ ]:
import os
from agentic_sentiment.colab_sync import push_repo_changes
from agentic_sentiment.eval.results_log import format_results_entry

# Document this run in the repo immediately, straight to main --
# the checkpoint itself stays on Drive (that's what RunDir is for),
# this is just the durable pointer to it, plus the eval metrics
# trainer.evaluate() just produced. Use a fine-grained PAT scoped to
# just this repo (Contents: read/write), not a classic "repo"-scoped
# token -- set it in Colab Secrets as GITHUB_TOKEN. Unlike HF_TOKEN
# (which must raise loudly if missing, since training can't proceed
# without it), a missing GITHUB_TOKEN is fine here -- this step is
# optional documentation, so degrade to a print instead.
try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = os.environ.get("GITHUB_TOKEN", "")

row = run_dir.history_row("phase1")
entry = format_results_entry(
    run_label="Phase 1 LoRA retrain",
    context={"dataset": "Kaggle arhamrumi/amazon-product-reviews", "run_id": run_dir.run_id},
    metrics_by_name={"phase1": metrics},
)
if not github_token:
    print("GITHUB_TOKEN not set -- skipping auto-push. Report for this run:")
    print(row)
    print(entry)
else:
    # Only guard the FILE APPENDS against duplication -- always call
    # push_repo_changes regardless, so a retry after a push that
    # succeeded locally (commit made) but failed to actually push
    # still gets pushed, instead of silently doing nothing because
    # the row/entry already looked "recorded."
    history_path = "/content/repo/docs/CHECKPOINT_HISTORY.md"
    existing_history = open(history_path).read() if os.path.exists(history_path) else ""
    if row not in existing_history:
        with open(history_path, "a") as f:
            f.write(row + "\n")

    results_path = "/content/repo/docs/RESULTS.md"
    existing_results = open(results_path).read() if os.path.exists(results_path) else ""
    if entry not in existing_results:
        with open(results_path, "a") as f:
            f.write(entry)

    print(push_repo_changes(
        repo_dir="/content/repo", paths=["docs/CHECKPOINT_HISTORY.md", "docs/RESULTS.md"],
        message=f"Phase 1 training run {run_dir.run_id}", token=github_token,
    ))